In [13]:
import numpy as np
import pandas as pd

TRAIN_LABEL = 'data/labeledTrainData.tsv'
TRAIN_UNLABEL = 'data/unlabeledTrainData.tsv'
TEST = 'data/testData.tsv'

all_data = pd.read_csv(TRAIN_LABEL, header=0, delimiter="\t", quoting=3)
bonus_data = pd.read_csv(TRAIN_UNLABEL, header=0, delimiter="\t", quoting=3)
test = pd.read_csv(TEST, header=0, delimiter="\t", quoting=3)

print(all_data.shape)
print(bonus_data.shape)
print(test.shape)

print(all_data.head(2))

all_data.drop(columns=['id'], inplace=True)
bonus_data.drop(columns=['id'], inplace=True)

(25000, 3)
(50000, 2)
(25000, 2)
         id  sentiment                                             review
0  "5814_8"          1  "With all this stuff going down at the moment ...
1  "2381_9"          1  "\"The Classic War of the Worlds\" by Timothy ...


In [14]:
import re

def clean_text(text):
    text = text.lower()
    cleaned = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

all_data['review'] = all_data['review'].apply(clean_text)
bonus_data['review'] = bonus_data['review'].apply(clean_text)
test['review'] = test['review'].apply(clean_text)

test['id'] = [x.replace('"', '') for x in test['id']]
print(test.head(2))

         id                                             review
0  12311_10  naturally in a film whos main themes are of mo...
1    8348_2  this movie is a disaster within a disaster fil...


In [15]:
from sklearn.model_selection import train_test_split

train, val = train_test_split(all_data, test_size=0.2, random_state=42, shuffle=True)

In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer

NUM_FEATURES = 10000

In [17]:
vectorizer = TfidfVectorizer(max_features=NUM_FEATURES)

X_train = vectorizer.fit_transform(train['review']).toarray()
y_train = train['sentiment']

X_val = vectorizer.transform(val['review']).toarray()
y_val = val['sentiment']

X_test = vectorizer.transform(test['review']).toarray()

In [18]:
print(X_train)

[[0.         0.05118917 0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]]


In [19]:
vocab = vectorizer.get_feature_names_out()
print(f'Vocabulary size: {len(vocab)}')

Vocabulary size: 10000


In [ ]:
forest = RandomForestClassifier(random_state=42, n_jobs=-1)

grid_params = {
    'n_estimators': [100, 200, 400],
}

model = GridSearchCV(
    estimator=forest,
    param_grid=grid_params,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1,
    verbose=2
)

model.fit(X_train, y_train)

val_preds = model.predict(X_val)

auc = roc_auc_score(y_val, val_preds)
print(f'Validation AUC: {auc}')

Fitting 3 folds for each of 81 candidates, totalling 243 fits


In [ ]:
vectorizer = TfidfVectorizer(max_features=NUM_FEATURES)
X_train = vectorizer.fit_transform(all_data['review']).toarray()
y_train = all_data['sentiment']
X_test = vectorizer.transform(test['review']).toarray()

model = RandomForestClassifier(random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

test_preds = model.predict(X_test)

submission = pd.DataFrame({
    'id': test['id'],
    'sentiment': test_preds
})

submission.to_csv('submissions/tfidf_rf.csv', index=False)